# 08. RAG 품질 평가 - AI 트렌드 보고서 기반

## 실습 목표

이번 실습에서는 **AI 트렌드 보고서 기반 RAG**의 품질을 평가한다.

RAG 평가는 최종 답변만 보는 것이 아니라, `검색 단계(Retrieval)`, `답변 생성 단계(Generation)`, `전체 결과 평가(End-to-End)`로 나누어 확인하는 것이 중요하다.

| 평가 영역 | 평가 항목 | 확인 내용 | 사용하는 RAG 수행 결과 |해석|
|---|---|---|---|---|
| Retrieval | Context Hit | 검색 문서에 정답에 필요한 핵심 키워드가 포함되었는가 | `docs` |낮으면 검색 단계 문제 가능성|
| Generation | Faithfulness | 답변이 검색 문서에 근거하는가 | `answer`, `docs` |낮으면 환각 가능성|
| Generation | Answer Relevancy | 답변이 질문에 맞는가 | `question`, `answer` |낮으면 답변 초점 문제점|
| End-to-End | Answer Correctness | 답변이 기준 답변과 의미적으로 일치하는가 | `ground_truth`, `answer` |낮으면 전체 RAG 품질 문제|

핵심 원칙은 다음과 같다.

```text
질문 1개
  ↓
RAG 1회 수행
  - VectorDB 검색
  - 검색 문서 기반 답변 생성
  ↓
동일한 RAG 수행 결과(answer, docs)
  ↓
4가지 평가를 모두 계산
```

> 이 노트북은 `02_data_pdf_to_md.ipynb`에서 만든 Markdown 기반 ChromaDB를 사용한다.


## 1. 실습 환경 준비

아래 코드는 앞 실습에서 만든 AI 트렌드 보고서 Vector Store를 불러온다.

사용하는 Vector Store 정보는 다음과 같다.

```python
collection_name="ai_trend_report_2026_md"
persist_directory="chroma_db/ai_trend_report_md"
```


In [1]:
from pathlib import Path

from dotenv import load_dotenv
load_dotenv(override=True, dotenv_path="../.env")

from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_chroma import Chroma
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

# 02_data_pdf_to_md.ipynb에서 생성한 Markdown 기반 ChromaDB 불러오기
DB_PATH = Path("chroma_db/ai_trend_report_md")
COLLECTION_NAME = "ai_trend_report_2026_md"

if not DB_PATH.exists():
    raise FileNotFoundError(
        f"ChromaDB 폴더를 찾을 수 없습니다: {DB_PATH}\n"
        "먼저 02_data_pdf_to_md.ipynb를 실행하세요."
    )

vector_store = Chroma(
    collection_name=COLLECTION_NAME,
    embedding_function=embeddings,
    persist_directory=str(DB_PATH)
)

# 기본 검색기: 질문당 관련 문서 4개 검색
retriever = vector_store.as_retriever(search_kwargs={"k": 4})

print("Vector Store 준비 완료")
print("저장된 문서 수:", vector_store._collection.count())


Vector Store 준비 완료
저장된 문서 수: 29


## 2. 기본 RAG 답변 함수 만들기

초보자가 흐름을 보기 쉽도록 RAG 실행 과정을 함수 하나로 정리한다.

```text
질문 입력
  ↓
VectorDB에서 관련 문서 검색
  ↓
검색 문서를 context로 결합
  ↓
question + context를 프롬프트에 입력
  ↓
LLM 답변 생성
  ↓
answer, docs 반환
```

평가에서는 **동일한 RAG 수행 결과인 `answer`, `docs`**를 모든 평가 지표가 함께 사용한다.


In [2]:
def format_docs(docs):
    """검색된 Document들을 하나의 context 문자열로 합친다."""
    return "\n\n".join(doc.page_content for doc in docs)


def doc_label(doc):
    """검색 문서의 metadata를 보기 좋게 정리한다."""
    metadata = doc.metadata

    source = metadata.get("source", "?")
    chunk_id = metadata.get("chunk_id", "?")
    doc_type = metadata.get("doc_type", "?")
    source_name = metadata.get("source_name", "?")

    return f"chunk_id={chunk_id} | doc_type={doc_type} | source_name={source_name} | source={source}"


answer_prompt = ChatPromptTemplate.from_messages([
    ("system", """아래 문서에 근거해서만 질문에 답하세요.
문서에서 확인할 수 없는 내용은 '문서에서 확인할 수 없습니다'라고 답하세요.
한 줄에 한 문장씩 깔끔하게 정리하세요.

[문서]
{context}"""),
    ("human", "{question}")
])


def rag_answer(question: str, retriever, verbose: bool = False):
    """
    질문을 입력하면 RAG 답변과 검색된 문서를 함께 반환한다.

    전체 평가에서는 이 함수가 만든 동일한 RAG 수행 결과(answer, docs)를 기준으로
    Context Hit, Faithfulness, Answer Relevancy, Answer Correctness를 모두 계산한다.
    """
    # 1. VectorDB 검색
    docs = retriever.invoke(question)

    # 2. 검색 문서를 context로 정리
    context = format_docs(docs)

    if verbose:
        print("\n[검색된 문서]")
        for i, doc in enumerate(docs, start=1):
            print(f"문서 {i}: {doc_label(doc)}")

    # 3. 검색 문서 기반 답변 생성
    chain = answer_prompt | llm | StrOutputParser()
    answer = chain.invoke({
        "question": question,
        "context": context
    })

    return answer, docs


# 간단 테스트
test_question = "2026년 AI 기술 전망에서 핵심 변화는 무엇인가요?"
test_answer, test_docs = rag_answer(test_question, retriever, verbose=True)

print("\n[질문]")
print(test_question)

print("\n[답변]")
print(test_answer)

print("\n검색된 문서 수:", len(test_docs))



[검색된 문서]
문서 1: chunk_id=21 | doc_type=markdown | source_name=AI@Data_Report_CLEANED.md | source=data\AI@Data_Report_CLEANED.md
문서 2: chunk_id=6 | doc_type=markdown | source_name=AI@Data_Report_CLEANED.md | source=data\AI@Data_Report_CLEANED.md
문서 3: chunk_id=23 | doc_type=markdown | source_name=AI@Data_Report_CLEANED.md | source=data\AI@Data_Report_CLEANED.md
문서 4: chunk_id=4 | doc_type=markdown | source_name=AI@Data_Report_CLEANED.md | source=data\AI@Data_Report_CLEANED.md

[질문]
2026년 AI 기술 전망에서 핵심 변화는 무엇인가요?

[답변]
2026년 AI 기술 전망에서 핵심 변화는 기능 고도화입니다.  
기술 적용이 클라우드에서 온디바이스로 확대되는 양상이 나타날 것으로 예상됩니다.  
고위험군 평가와 책임 기준 명료화가 진행될 것입니다.  
안전성·정합성 검증의 중요성이 커질 것으로 전망됩니다.  
데이터 품질 표준화와 모델 신뢰성 평가 기준 구축이 요구됩니다.

검색된 문서 수: 4


## 3. 평가 데이터셋 준비

평가 데이터셋은 질문, 기준 답변, 검색 확인용 핵심 키워드로 구성한다.

- `question`: RAG에 입력할 질문
- `ground_truth`: 기대하는 기준 답변
- `expected_keywords`: 검색된 문서에 포함되어야 할 핵심 키워드

이번 실습에서는 `ground_truth`를 **Answer Correctness 평가**에 사용한다.


In [3]:
eval_dataset = [
    {
        "question": "본 보고서의 분석 목적은 무엇인가요?",
        "ground_truth": (
            "본 보고서는 2025년 국내외 주요 매체에서 수집한 AI 관련 텍스트를 바탕으로 "
            "산업·기술·정책 세 영역에서 형성되는 담론 구조와 주요 흐름을 조망하고, "
            "LDA 토픽모델링을 통해 반복적으로 등장하는 핵심 논점과 의미 축을 도출함으로써 "
            "2026년 AI 생태계를 전망하기 위한 기초 자료를 제공하는 것을 목적으로 한다."
        ),
        "expected_keywords": [
            "AI 관련 텍스트",
            "산업",
            "기술",
            "정책",
            "담론 구조",
            "2026년 AI 생태계"
        ]
    },
    {
        "question": "산업·기술·정책 분야별 데이터 수집 키워드는 각각 무엇인가요?",
        "ground_truth": (
            "AI 산업 분야는 AI투자, 시장규모, 기업도입, 생성형AI, 펀딩, 스타트업, 산업전환을 사용하고, "
            "AI 기술 분야는 AI에이전트, 멀티모달, LLM, 온디바이스AI, 오픈소스, 딥시크, 모델개발을 사용하며, "
            "AI 정책 분야는 AI규제, AI기본법, EU AI Act, 거버넌스, 행정명령, 데이터보호, 인프라정책을 사용한다."
        ),
        "expected_keywords": [
            "AI투자", "시장규모",
            "AI에이전트", "멀티모달",
            "AI규제", "EU AI Act"
        ]
    },
    {
        "question": "AI 산업, AI 기술, AI 정책의 핵심 토픽은 각각 무엇인가요?",
        "ground_truth": (
            "AI 산업의 핵심 토픽은 시장 규모 확대와 비즈니스 구조 전환이고, "
            "AI 기술의 핵심 토픽은 기능 고도화와 적용 범위 확장이며, "
            "AI 정책의 핵심 토픽은 안전성과 책임 중심의 규제 거버넌스이다."
        ),
        "expected_keywords": ["시장 규모", "기능 고도화", "안전성과 책임", "규제 거버넌스"]
    },
    {
        "question": "2026년 AI 기술 전망에서 핵심 변화는 무엇인가요?",
        "ground_truth": (
            "2026년 AI 기술은 지능 구조 고도화와 데이터 한계 보완을 중심으로 진화할 전망이며, "
            "합성데이터, 추론형 AI, 멀티모달 기술이 주요 경쟁 축으로 부상하고 "
            "신뢰성 평가와 표준화의 중요성이 커질 것으로 제시된다."
        ),
        "expected_keywords": ["지능 구조 고도화", "합성데이터", "추론형 AI", "멀티모달", "신뢰성"]
    },
    {
        "question": "AI 정책 전망에서 안전성과 책임은 왜 중요하게 다루어지나요?",
        "ground_truth": (
            "AI 확산이 빨라지면서 고위험 분야의 책임과 안전성 확보, 데이터·저작권 정책 명확화, "
            "국제 규제와의 정합성이 중요해지고 있으며, 이를 통해 예측 가능한 정책 생태계와 "
            "신뢰 기반 거버넌스를 구축할 필요가 있기 때문이다."
        ),
        "expected_keywords": ["안전성", "책임", "고위험", "저작권", "거버넌스"]
    },
    {
        "question": "추론형 데이터는 어떤 유형으로 구분되나요?",
        "ground_truth": "추론형 데이터는 단계적 과정 추론, 근거 기반 설명, 맥락·관계 이해, 논리·취약점 검증으로 구분된다.",
        "expected_keywords": ["단계적", "근거", "맥락", "논리", "취약점"]
    },
]

print(f"평가 데이터셋 수: {len(eval_dataset)}개")


평가 데이터셋 수: 6개


## 4. 평가 점수 파싱 함수

LLM 평가자는 숫자만 출력하도록 지시하더라도, 실제로는 `0.8점`, `Score: 0.8`처럼 답할 수 있다.

따라서 점수 문자열에서 숫자만 안전하게 추출하는 함수를 먼저 만든다.


In [4]:
def parse_score(score_text: str) -> float:
    """LLM이 출력한 점수 문자열에서 숫자를 추출하고 0.0~1.0 범위로 제한한다."""
    import re

    match = re.search(r"\d+(\.\d+)?", score_text)
    if match:
        score = float(match.group())
        return max(0.0, min(score, 1.0))

    return 0.0


## 5. 평가 함수 만들기

아래 4가지 평가 함수는 모두 **이미 수행된 RAG 결과**를 사용한다.

즉, 평가 함수 내부에서 VectorDB를 다시 검색하지 않는다.

```text
rag_answer(question, retriever)
  ↓
answer, docs
  ↓
4가지 평가 함수에 전달
```


### 5.1 Faithfulness

Faithfulness는 **답변이 검색된 문서에 근거하고 있는지**를 평가한다.

```text
answer + docs
  ↓
LLM 평가자
  ↓
0.0~1.0 점수
```


In [5]:
faithfulness_prompt = ChatPromptTemplate.from_messages([
    ("system", """다음 답변이 주어진 AI 트렌드 보고서 컨텍스트에 근거하는지 평가하세요.

평가 기준:
- 답변이 컨텍스트에 있는 내용에 근거하면 높은 점수를 주세요.
- 컨텍스트에 없는 내용을 만들어내면 낮은 점수를 주세요.
- 컨텍스트에 근거가 없어서 '문서에서 확인할 수 없습니다'라고 답한 경우는 환각이 아니므로 높은 점수를 줄 수 있습니다.
- 단, 답변이 질문과 무관하거나 컨텍스트와 맞지 않으면 낮은 점수를 주세요.

0.0~1.0 사이의 숫자만 출력하세요."""),
    ("human", "컨텍스트:\n{context}\n\n답변:\n{answer}")
])

faithfulness_chain = faithfulness_prompt | llm | StrOutputParser()


def evaluate_faithfulness(answer: str, docs) -> float:
    """
    RAG 답변 생성에 실제 사용된 docs를 기준으로 Faithfulness를 평가한다.

    이 함수는 VectorDB를 새로 검색하지 않는다.
    """
    context = format_docs(docs)

    score_text = faithfulness_chain.invoke({
        "context": context,
        "answer": answer
    })

    return parse_score(score_text)


In [6]:
# 질문 테스트
item = eval_dataset[0]
# 1. 벡터DB 검색 + RAG 답변 생성
answer, docs = rag_answer(item["question"], retriever)
# 2. llm답변과 검색된 문서의 근거 일치 평가
result = evaluate_faithfulness(answer, docs)

print("[질문]")
print( item["question"])

print("\n[답변]")
print(answer)

print("\n[Faithfulness 점수]")
print(result)

print("\n[검색된 문서 수]")
print(len(docs))

[질문]
본 보고서의 분석 목적은 무엇인가요?

[답변]
본 보고서의 분석 목적은 AI 트렌드의 전체 맥락과 세부 이슈를 포괄적으로 이해하고, 이를 기반으로 국가 정책, 기업 전략, 기술 개발 방향 설정에 활용 가능한 실질적 인사이트를 제공하는 것입니다. 

또한, 중장기 전략 수립을 위한 근거를 제공하고, 정책 담당자에게는 법적 기반 마련 시 고려해야 할 우선순위를 제시하며, 기업에는 투자·사업 포트폴리오 조정 방향을, 연구기관에는 후속 연구가 필요한 공백 영역을 제시하는 것을 목표로 합니다.

[Faithfulness 점수]
1.0

[검색된 문서 수]
4


### 5.2 Answer Relevancy

Answer Relevancy는 **답변이 질문에 직접적으로 답하고 있는지**를 평가한다.

```text
question + answer
  ↓
LLM 평가자
  ↓
0.0~1.0 점수
```


In [7]:
relevancy_prompt = ChatPromptTemplate.from_messages([
    ("system", """다음 답변이 질문에 얼마나 직접적으로 답하고 있는지 평가하세요.

평가 기준:
- 질문에서 묻는 핵심 내용에 직접 답하면 높은 점수를 주세요.
- 질문과 관련은 있지만 핵심 답을 제공하지 못하면 낮은 점수를 주세요.
- 질문과 다른 내용을 설명하면 낮은 점수를 주세요.
- '문서에서 확인할 수 없습니다'처럼 실제 답을 제공하지 못한 경우에는 낮은 점수를 주세요.

0.0~1.0 사이의 숫자만 출력하세요."""),
    ("human", "질문:\n{question}\n\n답변:\n{answer}")
])

relevancy_chain = relevancy_prompt | llm | StrOutputParser()


def evaluate_relevancy(question: str, answer: str) -> float:
    """
    RAG 수행 결과로 생성된 answer가 question에 직접적으로 답하는지 평가한다.

    이 함수는 VectorDB를 새로 검색하지 않는다.
    """
    score_text = relevancy_chain.invoke({
        "question": question,
        "answer": answer
    })

    return parse_score(score_text)


In [8]:
# 질문 테스트
item = eval_dataset[0]
# 1. 벡터DB 검색 + RAG 답변 생성
answer, docs = rag_answer(item["question"], retriever)
# 2. 질문에 대한 답변의 직접적 관련성 평가
result = evaluate_relevancy(item["question"], answer)

print("[질문]")
print( item["question"])

print("\n[답변]")
print(answer)

print("\n[Relevancy 점수]")
print(result)

print("\n[검색된 문서 수]")
print(len(docs))

[질문]
본 보고서의 분석 목적은 무엇인가요?

[답변]
본 보고서의 분석 목적은 AI 트렌드의 전체 맥락과 세부 이슈를 포괄적으로 이해하고, 이를 기반으로 국가 정책, 기업 전략, 기술 개발 방향 설정에 활용 가능한 실질적 인사이트를 제공하는 것입니다. 

또한, 중장기 전략 수립을 위한 근거를 제공하고, 정책 담당자에게는 법적 기반 마련 시 고려해야 할 우선순위를 제시하며, 기업에는 투자·사업 포트폴리오 조정 방향을, 연구기관에는 후속 연구가 필요한 공백 영역을 제시하는 것을 목표로 합니다.

[Relevancy 점수]
1.0

[검색된 문서 수]
4


### 5.3 Answer Correctness

Answer Correctness는 **생성 답변이 기준 답변과 의미적으로 일치하는지**를 평가한다.

```text
ground_truth + answer
  ↓
LLM 평가자
  ↓
0.0~1.0 점수
```


In [9]:
correctness_prompt = ChatPromptTemplate.from_messages([
    ("system", """다음 생성 답변이 기준 답변과 의미적으로 얼마나 일치하는지 평가하세요.

평가 기준:
- 핵심 의미가 같으면 높은 점수
- 중요한 내용을 빠뜨리면 감점
- 기준 답변에 없는 내용을 과도하게 추가하면 감점
- 표현이 달라도 의미가 같으면 인정

0.0~1.0 사이의 숫자만 출력하세요."""),
    ("human", """[기준 답변]
{ground_truth}

[생성 답변]
{answer}""")
])

correctness_chain = correctness_prompt | llm | StrOutputParser()


def evaluate_correctness(ground_truth: str, answer: str) -> float:
    """
    RAG 수행 결과로 생성된 answer가 기준 답변과 의미적으로 일치하는지 평가한다.

    이 함수는 VectorDB를 새로 검색하지 않는다.
    """
    score_text = correctness_chain.invoke({
        "ground_truth": ground_truth,
        "answer": answer
    })

    return parse_score(score_text)


In [10]:
# 질문 테스트
item = eval_dataset[0]
# 1. 벡터DB 검색 + RAG 답변 생성
answer, docs = rag_answer(item["question"], retriever)
# 2. 기준 답변과 생성 답변의 의미적 일치 평가
result = evaluate_relevancy(item["ground_truth"], answer)

print("[기준 답변]")
print(item["ground_truth"])

print("\n[LLM 답변]")
print(answer)

print("\n[answer correctness 점수]")
print(result)

print("\n[검색된 문서 수]")
print(len(docs))

[기준 답변]
본 보고서는 2025년 국내외 주요 매체에서 수집한 AI 관련 텍스트를 바탕으로 산업·기술·정책 세 영역에서 형성되는 담론 구조와 주요 흐름을 조망하고, LDA 토픽모델링을 통해 반복적으로 등장하는 핵심 논점과 의미 축을 도출함으로써 2026년 AI 생태계를 전망하기 위한 기초 자료를 제공하는 것을 목적으로 한다.

[LLM 답변]
본 보고서의 분석 목적은 AI 트렌드의 전체 맥락과 세부 이슈를 포괄적으로 이해하고, 이를 기반으로 국가 정책, 기업 전략, 기술 개발 방향 설정에 활용 가능한 실질적 인사이트를 제공하는 것입니다. 

또한, 중장기 전략 수립을 위한 근거를 제공하고, 정책 담당자에게는 법적 기반 마련 시 고려해야 할 우선순위를 제시하며, 기업에는 투자·사업 포트폴리오 조정 방향을 제안하는 것을 목표로 합니다. 

마지막으로, 연구기관에는 후속 정량·정성 연구가 필요한 공백 영역을 제시하는 것을 포함합니다.

[answer correctness 점수]
0.8

[검색된 문서 수]
4


### 5.4 Context Hit

Context Hit는 **RAG 답변 생성에 실제 사용된 검색 문서에 정답 키워드가 포함되어 있는지**를 확인한다.

```text
item + docs
  ↓
expected_keywords 포함 여부 확인
  ↓
True / False
```

단순 문자열 비교는 띄어쓰기 차이에 약하므로, 공백과 일부 기호를 제거하는 간단한 정규화를 적용한다.


In [11]:
def normalize_text(text: str) -> str:
    """간단한 문자열 비교를 위해 공백과 일부 기호를 제거한다."""
    return (
        text.lower()
        .replace(" ", "")
        .replace("·", "")
        .replace("-", "")
        .replace("_", "")
    )


def evaluate_context_hit_from_docs(item: dict, docs, min_matches: int = 2):
    """
    RAG 답변 생성에 실제 사용된 docs에 expected_keywords가 충분히 포함되어 있는지 확인한다.

    이 함수는 VectorDB를 새로 검색하지 않는다.
    """
    context = format_docs(docs)
    norm_context = normalize_text(context)

    matched_keywords = []

    for keyword in item["expected_keywords"]:
        if normalize_text(keyword) in norm_context:
            matched_keywords.append(keyword)

    hit = len(matched_keywords) >= min_matches

    return hit, matched_keywords


In [12]:
# 질문 테스트
item = eval_dataset[0]
# 1. 벡터DB 검색 + RAG 답변 생성
answer, docs = rag_answer(item["question"], retriever)
# 2. 기준 답변과 생성 답변의 의미적 일치 평가
result = evaluate_context_hit_from_docs(item, docs)

print("[질문 관련 키워드]")
print(item["expected_keywords"])

print("\n[LLM 답변 키워드]")
print(result[1])

print("\n[context hit 점수]")
print(result[0])

print("\n[검색된 문서 수]")
print(len(docs))

[질문 관련 키워드]
['AI 관련 텍스트', '산업', '기술', '정책', '담론 구조', '2026년 AI 생태계']

[LLM 답변 키워드]
['산업', '기술', '정책', '담론 구조']

[context hit 점수]
True

[검색된 문서 수]
4


## 6. 개별 문항 테스트

아래 코드는 하나의 문항에 대해 RAG를 **한 번만 수행**하고, 그 동일한 결과로 4가지 평가를 모두 계산한다.


In [13]:
# 테스트 문항 선택
sample_item = eval_dataset[0]

# 1. RAG 1회 수행
sample_answer, sample_docs = rag_answer(
    sample_item["question"],
    retriever,
    verbose=True
)

# 2. 동일한 RAG 수행 결과로 4가지 평가 계산
sample_context_hit, sample_matched_keywords = evaluate_context_hit_from_docs(sample_item, sample_docs)
sample_faithfulness = evaluate_faithfulness(sample_answer, sample_docs)
sample_relevancy = evaluate_relevancy(sample_item["question"], sample_answer)
sample_correctness = evaluate_correctness(sample_item["ground_truth"], sample_answer)

print("\n[질문]")
print(sample_item["question"])

print("\n[답변]")
print(sample_answer)

print("\n[평가 결과]")
print(f"Context Hit: {sample_context_hit}")
print(f"Matched Keywords: {sample_matched_keywords}")
print(f"Faithfulness: {sample_faithfulness:.2f}")
print(f"Answer Relevancy: {sample_relevancy:.2f}")
print(f"Answer Correctness: {sample_correctness:.2f}")



[검색된 문서]
문서 1: chunk_id=9 | doc_type=markdown | source_name=AI@Data_Report_CLEANED.md | source=data\AI@Data_Report_CLEANED.md
문서 2: chunk_id=12 | doc_type=markdown | source_name=AI@Data_Report_CLEANED.md | source=data\AI@Data_Report_CLEANED.md
문서 3: chunk_id=8 | doc_type=markdown | source_name=AI@Data_Report_CLEANED.md | source=data\AI@Data_Report_CLEANED.md
문서 4: chunk_id=7 | doc_type=markdown | source_name=AI@Data_Report_CLEANED.md | source=data\AI@Data_Report_CLEANED.md

[질문]
본 보고서의 분석 목적은 무엇인가요?

[답변]
본 보고서의 분석 목적은 AI 트렌드의 전체 맥락과 세부 이슈를 포괄적으로 이해하고, 이를 기반으로 국가 정책, 기업 전략, 기술 개발 방향 설정에 활용 가능한 실질적 인사이트를 제공하는 것입니다.  
또한, 중장기 전략 수립을 위한 근거를 제공하고, 정책 담당자, 기업, 연구기관에 필요한 정보를 제시하는 것을 목표로 합니다.

[평가 결과]
Context Hit: True
Matched Keywords: ['산업', '기술', '정책', '담론 구조']
Faithfulness: 1.00
Answer Relevancy: 1.00
Answer Correctness: 0.40


## 7. 전체 평가 실행

평가 데이터셋 전체에 대해 4가지 평가 값을 계산한다.

중요한 원칙은 **질문 하나당 RAG를 한 번만 수행**하고, 그 결과로 나온 동일한 `answer`, `docs`를 모든 평가에 함께 사용하는 것이다.


결과와 함께 확인하기

In [15]:
import pandas as pd


def evaluate_rag_result(item: dict, retriever, verbose: bool = False) -> dict:
    """
    하나의 평가 문항에 대해 RAG를 한 번 수행하고,
    그 동일한 수행 결과(answer, docs)를 기준으로 4가지 평가를 계산한다.
    """
    question = item["question"]

    # 1. RAG 수행: VectorDB 검색 + 검색 문서 기반 답변 생성
    answer, docs = rag_answer(question, retriever, verbose=verbose)

    # 2. Retrieval 평가: 검색된 docs가 필요한 근거를 포함하는지 확인
    context_hit, matched_keywords = evaluate_context_hit_from_docs(item, docs)

    # 3. Generation 평가: 같은 answer와 docs를 기준으로 평가
    faithfulness = evaluate_faithfulness(answer, docs)
    relevancy = evaluate_relevancy(question, answer)

    # 4. End-to-End 평가: 같은 answer와 기준 답변을 비교
    correctness = evaluate_correctness(item["ground_truth"], answer)

    return {
        "question": question,
        "faithfulness": faithfulness,
        "relevancy": relevancy,
        "correctness": correctness,
        "context_hit": context_hit,
        "matched_keywords": ", ".join(matched_keywords),
        "answer_preview": answer[:120],
        "num_docs": len(docs)
    }


results = []

for item in eval_dataset:
    result = evaluate_rag_result(item, retriever, verbose=False)
    results.append(result)

    print(f"Q. {result['question']}")
    print(
        f"   Faithfulness={result['faithfulness']:.2f} | "
        f"Relevancy={result['relevancy']:.2f} | "
        f"Correctness={result['correctness']:.2f} | "
        f"Context Hit={result['context_hit']}"
    )
    print(f"   Matched Keywords: {result['matched_keywords']}")
    print("-" * 80)

df_results = pd.DataFrame(results)

print("\n[평균 평가 결과]")
print(f"평균 Faithfulness: {df_results['faithfulness'].mean():.2f}")
print(f"평균 Answer Relevancy: {df_results['relevancy'].mean():.2f}")
print(f"평균 Answer Correctness: {df_results['correctness'].mean():.2f}")
print(f"Context Hit Rate: {df_results['context_hit'].mean():.2f}")

df_results


Q. 본 보고서의 분석 목적은 무엇인가요?
   Faithfulness=1.00 | Relevancy=1.00 | Correctness=0.40 | Context Hit=True
   Matched Keywords: 산업, 기술, 정책, 담론 구조
--------------------------------------------------------------------------------
Q. 산업·기술·정책 분야별 데이터 수집 키워드는 각각 무엇인가요?
   Faithfulness=1.00 | Relevancy=1.00 | Correctness=1.00 | Context Hit=True
   Matched Keywords: AI투자, 시장규모, AI에이전트, 멀티모달, AI규제, EU AI Act
--------------------------------------------------------------------------------
Q. AI 산업, AI 기술, AI 정책의 핵심 토픽은 각각 무엇인가요?
   Faithfulness=1.00 | Relevancy=1.00 | Correctness=1.00 | Context Hit=True
   Matched Keywords: 시장 규모, 기능 고도화, 안전성과 책임
--------------------------------------------------------------------------------
Q. 2026년 AI 기술 전망에서 핵심 변화는 무엇인가요?
   Faithfulness=1.00 | Relevancy=1.00 | Correctness=0.60 | Context Hit=True
   Matched Keywords: 지능 구조 고도화, 합성데이터, 추론형 AI, 멀티모달, 신뢰성
--------------------------------------------------------------------------------
Q. AI 정책 전망에서 안전성과 책임은 왜 중요하게 다루

,question,faithfulness,relevancy,correctness,context_hit,matched_keywords,answer_preview,num_docs
0,본 보고서의 분석 목적은 무엇인가요?,1.0,1.0,0.4,True,"산업, 기술, 정책, 담론 구조",본 보고서의 분석 목적은 AI 트렌드의 전체 맥락과 세부 이슈를 포괄적으로 이해하고...,4
1,산업·기술·정책 분야별 데이터 수집 키워드는 각각 무엇인가요?,1.0,1.0,1.0,True,"AI투자, 시장규모, AI에이전트, 멀티모달, AI규제, EU AI Act",산업·기술·정책 분야별 데이터 수집 키워드는 다음과 같습니다.\n\nAI산업: AI...,4
2,"AI 산업, AI 기술, AI 정책의 핵심 토픽은 각각 무엇인가요?",1.0,1.0,1.0,True,"시장 규모, 기능 고도화, 안전성과 책임",AI 산업의 핵심 토픽은 시장 규모 확대와 비즈니스 구조 전환입니다. \nAI 기...,4
3,2026년 AI 기술 전망에서 핵심 변화는 무엇인가요?,1.0,1.0,0.6,True,"지능 구조 고도화, 합성데이터, 추론형 AI, 멀티모달, 신뢰성",2026년 AI 기술 전망에서 핵심 변화는 기능 고도화입니다. \n기술 적용이 클...,4
4,AI 정책 전망에서 안전성과 책임은 왜 중요하게 다루어지나요?,1.0,1.0,0.7,True,"안전성, 책임, 고위험, 저작권, 거버넌스",AI 정책에서 안전성과 책임은 핵심 이슈로 부각되고 있습니다. \n\n이는 AI 확...,4
5,추론형 데이터는 어떤 유형으로 구분되나요?,1.0,1.0,1.0,True,"단계적, 근거, 맥락, 논리, 취약점",추론형 데이터는 4가지 유형으로 구분됩니다. \n첫 번째는 단계적 '과정' 추론입...,4


## 8. Pass / Fail 기준 추가

실무에서는 평균 점수만 보지 않고, 각 질문별로 **배포 가능 여부**나 **개선 필요 여부**를 판단한다.

아래 기준은 수업용 예시이다.  
실제 서비스에서는 도메인, 위험도, 사용자 요구 수준에 따라 기준을 별도로 정해야 한다.


In [16]:
THRESHOLDS = {
    "faithfulness": 0.8,
    "relevancy":    0.8,
    "correctness":  0.7,
    "context_hit":  0.6,
}

for r in results:
    passed = all(r[metric] >= threshold for metric, threshold in THRESHOLDS.items())
    status = "✅ PASS" if passed else "❌ FAIL"
    print(f"{status} | {r['question'][:35]}")
    if not passed:
        for metric, threshold in THRESHOLDS.items():
            if r[metric] < threshold:
                print(f"  → {metric}: {r[metric]:.2f} < {threshold} (기준 미달)")


❌ FAIL | 본 보고서의 분석 목적은 무엇인가요?
  → correctness: 0.40 < 0.7 (기준 미달)
✅ PASS | 산업·기술·정책 분야별 데이터 수집 키워드는 각각 무엇인가요?
✅ PASS | AI 산업, AI 기술, AI 정책의 핵심 토픽은 각각 무엇인가
❌ FAIL | 2026년 AI 기술 전망에서 핵심 변화는 무엇인가요?
  → correctness: 0.60 < 0.7 (기준 미달)
✅ PASS | AI 정책 전망에서 안전성과 책임은 왜 중요하게 다루어지나요?
✅ PASS | 추론형 데이터는 어떤 유형으로 구분되나요?


In [ ]:
def judge_pass_fail(row):
    """
    간단한 실습용 통과 기준.
    실제 서비스에서는 도메인별 기준을 별도로 설계해야 한다.
    """
    if row["faithfulness"] < 0.7:
        return "Fail: 문서 근거 부족"
    if row["relevancy"] < 0.7:
        return "Fail: 질문 적합성 낮음"
    if row["correctness"] < 0.7:
        return "Fail: 기준 답변과 불일치"
    if not row["context_hit"]:
        return "Fail: 검색 근거 부족"

    return "Pass"


df_results["decision"] = df_results.apply(judge_pass_fail, axis=1)

df_results[[
    "question",
    "faithfulness",
    "relevancy",
    "correctness",
    "context_hit",
    "decision"
]]


,question,faithfulness,relevancy,correctness,context_hit,decision
0,본 보고서의 분석 목적은 무엇인가요?,1.0,1.0,0.4,True,Fail: 기준 답변과 불일치
1,산업·기술·정책 분야별 데이터 수집 키워드는 각각 무엇인가요?,1.0,1.0,1.0,True,Pass
2,"AI 산업, AI 기술, AI 정책의 핵심 토픽은 각각 무엇인가요?",1.0,1.0,1.0,True,Pass
3,2026년 AI 기술 전망에서 핵심 변화는 무엇인가요?,1.0,1.0,0.5,True,Fail: 기준 답변과 불일치
4,AI 정책 전망에서 안전성과 책임은 왜 중요하게 다루어지나요?,1.0,1.0,0.8,True,Pass
5,추론형 데이터는 어떤 유형으로 구분되나요?,1.0,1.0,1.0,True,Pass


## 9. 검색 설정 비교: k 값 바꾸기

RAG 평가에서는 답변만 보는 것이 아니라, **검색 설정을 바꿨을 때 검색 품질이 어떻게 달라지는지**도 확인한다.

아래 예시는 `k=2`, `k=4`, `k=6`을 비교한다.

- `k=2`: 적은 문서만 검색하므로 빠르지만 필요한 근거가 누락될 수 있다.
- `k=4`: 기본 설정이다.
- `k=6`: 더 많은 문서를 검색하므로 근거를 찾을 가능성은 커지지만 관련 없는 문서도 섞일 수 있다.

주의할 점은 `k`를 키운다고 최종 답변 품질이 항상 좋아지는 것은 아니라는 점이다.  
필요한 근거가 늘어날 수도 있지만, 불필요한 노이즈도 함께 늘어날 수 있다.


In [ ]:
def evaluate_retriever_setting(eval_dataset, retriever, min_matches: int = 2):
    """특정 retriever 설정에서 Context Hit Rate를 계산한다."""
    hits = []

    for item in eval_dataset:
        docs = retriever.invoke(item["question"])
        hit, _ = evaluate_context_hit_from_docs(item, docs, min_matches=min_matches)
        hits.append(hit)

    return sum(hits) / len(hits)


retriever_settings = {
    "k=2": vector_store.as_retriever(search_kwargs={"k": 2}),
    "k=4": vector_store.as_retriever(search_kwargs={"k": 4}),
    "k=6": vector_store.as_retriever(search_kwargs={"k": 6}),
}

setting_results = []

for setting_name, test_retriever in retriever_settings.items():
    hit_rate = evaluate_retriever_setting(eval_dataset, test_retriever)
    setting_results.append({
        "setting": setting_name,
        "context_hit_rate": hit_rate
    })

df_setting_results = pd.DataFrame(setting_results)

df_setting_results


,setting,context_hit_rate
0,k=2,0.833333
1,k=4,1.000000
2,k=6,1.000000


## 10. 평가 결과 해석

평가 점수는 절대적인 진실이 아니라 **RAG 개선 방향을 찾기 위한 참고 지표**이다.

| 결과 패턴 | 해석 | 개선 방향 |
|---|---|---|
| Faithfulness가 낮음 | 답변이 검색 문서에 없는 내용을 포함했을 가능성이 있음 | 프롬프트 제약 강화, 출처 기반 답변 적용 |
| Answer Relevancy가 낮음 | 질문과 다른 방향으로 답변했을 가능성이 있음 | 질문 재작성, 답변 프롬프트 개선 |
| Answer Correctness가 낮음 | 기준 답변과 핵심 의미가 다를 가능성이 있음 | 검색 결과와 답변 생성 모두 점검 |
| Context Hit가 낮음 | 검색 단계에서 필요한 근거를 찾지 못했을 가능성이 있음 | Query Rewriting, Multi-Query, Hybrid Search 검토 |
| k를 키웠는데 Hit가 높아짐 | 기존 k가 너무 작아 근거가 누락되었을 수 있음 | k 증가 또는 Multi-Query 적용 |
| k를 키웠는데 답변이 산만해짐 | 관련 없는 문서가 함께 들어왔을 수 있음 | Reranking, Contextual Compression 적용 |

## 정리

```text
RAG 평가 기본 흐름

질문 준비
  ↓
RAG 수행
  - VectorDB 검색
  - 검색 docs 기반 답변 생성
  ↓
동일한 RAG 수행 결과(answer, docs) 확보
  ↓
Retrieval 평가
  - Context Hit: docs 기준
  ↓
Generation 평가
  - Faithfulness: answer + docs 기준
  - Answer Relevancy: question + answer 기준
  ↓
End-to-End 평가
  - Answer Correctness: ground_truth + answer 기준
  ↓
Pass / Fail 판단
  ↓
검색 설정 또는 Advanced RAG 기법 개선
```

이번 실습에서는 복잡한 평가 프레임워크를 사용하지 않고, 초보자가 이해하기 쉬운 방식으로 RAG 품질을 점검했다.

중요한 점은 RAG 평가를 하나의 점수로만 보지 않는 것이다.  
검색이 실패했는지, 답변 생성이 실패했는지, 기준 답변과 의미적으로 맞지 않는지를 나누어 확인해야 한다.


## 11. 실무 확장: RAG 평가 프레임워크

이번 실습에서는 평가 원리를 이해하기 위해 직접 간단한 평가 함수를 만들었다.  
실무에서는 다음과 같은 평가 도구를 함께 사용할 수 있다.

| 도구 | 특징 | 사용 상황 |
|---|---|---|
| Ragas | RAG 평가를 위한 오픈소스 프레임워크 | Faithfulness, relevancy, retrieval 품질을 자동 평가할 때 |
| DeepEval | LLM-as-a-judge 기반 평가 프레임워크 | 테스트 케이스 단위로 RAG 품질을 평가할 때 |
| LangSmith | LangChain 기반 앱 추적·평가·디버깅 도구 | RAG 실행 로그, 평가 결과, 실험 비교가 필요할 때 |

다만 초급 단계에서는 먼저 직접 평가 함수를 만들어 지표의 의미를 이해하는 것이 중요하다.  
도구는 나중에 붙일 수 있지만, 평가 관점이 정리되어 있지 않으면 결과를 해석하기 어렵다.
